# Report

NUS AY 20242025 Sem 2 CS5424

Group 10 

Benjamin Lee Jun Cheng A0286163Y

Lai Foong Ming A0268245X
 
This notebook is a report for the group project done for CS5424 Neural Networks and Deep Learning. 

### Use of AI Declaration 

Generative AI was used in this project to for debugging, troubleshooting and limited code generation. Additionally, a synthetic data generation experiment was conducted that is addressed at the end of the report. 

## Project Objectives


Context: 
- News media often presents information with varying degrees of political bias. Readers need reliable tools to quickly identify these biases, especially for breaking news, to make informed decisions about the content they consume. Current manual rating systems cannot keep pace with the volume and speed of modern news distribution.

Problem statement:
- Human-based news rating systems are not scalable
- Breaking news evaluated too late to be useful
- Push notifications and RSS feeds outpace manual review
- Rating sites have limited coverage of sources 

Approach: 
- Develop ML-based political bias classifier 

Goal: 
- Enable instant, on-demand bias assessment to enhance media literary 

## Project Flow

![project flow](images/ProjectFlow.png)

An initial run was conducted on the dataset provided by [Kaggle](https://www.kaggle.com/datasets/mayobanexsantana/political-bias/data). A subsequent run was then conducted with the parameters found in run 1. This isolated the impact of more diverse data on the performance of the individual models. 

## EDA

Dataset stats:
- n= 2185
- 5 target labels for bias: 
- Right, Lean Right, Centre, Lean Left, Left 
- News articles from 21 unique sites
- Heavily imbalance towards left

Analysis and Findings: 
- Source is a one to one mapping for Bias
- Analysis of text length and unique vocabulary shows that both are unlikely to be a good indicator with similar medians and IQR, as it will fail to discriminate strongly between adjacent classes
- Word cloud and LDA diagrams shows that topics covered and broadly the same across the political spectrum which is unsurprising. We expect all outlets to cover the same news story in different tonality/perspectives that give rise to the label. 

Pre-processing and next steps: 
- Drop Source due to being a 1:1 predictor of bias
- Remove source names from article text to remove bias indicator 
- For TD-IDF in decision tree and FFNN, apply stemming, remove numbers and punctuation 
- For LSTM, BERT and Llama, use full words (retain numbers and punctuation) 
- To deal with imbalance: 
Develop a scrapper in parallel to augment data for classes with lower representations 

For further commentary please see:
- EDA on the original kaggle dataset: [EDA.ipynb](EDA.ipynb)
- EDA on the scrapped allsides data: [EDA_allsides.ipynb](EDA_allsides.ipynb)
- EDA on the scrapped newsapi data: [EDA_newsapi.ipynb](EDA_newsapi.ipynb)
- EDA on the combined dataset: [EDA_combined.ipynb](EDA_combined.ipynb)
- EDA on the synthetic data: [EDA_syn.ipynb](EDA_syn.ipynb)

## Choice of Models

We chose decision tree to have a classic model for our baseline and FFNN for the baseline within the NN DL family of models. We chose LSTM over textCNN as we expect that the sequential or temporal nature of the data will play a large role in affecting the performance of the model. We chose BERT and Llama as representatives of their overall transformer and attention model families respectively as they can be run locally.  

## Models - Run 1 

1. Decision Tree [decisionTree.ipynb](decisionTree.ipynb)
2. FFNN(TD-IDF) [FFNN.ipynb](FFNN.ipynb)
3. FFNN (GloVe) [FFNN_GloVe.ipynb](FFNN_GloVe.ipynb)
4. LSTM [lstm.ipynb](lstm.ipynb)
5. BERT [bert.ipynb](bert.ipynb)
6. Llama (basic model) [llama_base.ipynb](llama_base.ipynb)
7. Llama (LoRA) [llama_lora.ipynb](llama_lora.ipynb)

### Performance Summary 

| Model            | Accuracy | Weighted F1 |
|------------------|----------|-------------|
| Decision Tree    | 59.27    | 60.29       |
| FFNN (TD-IDF)    | 69.79    | 68.85       |
| FFNN (GloVe)     | 32.27    | 33.72       |
| LSTM             | 37.76    | 41.46       |
| BERT             | 60.41    | 61.04       |
| TinyLlama        | 51.72    | 53.75       |
| TinyLlama (LoRA) | 64.76    | 63.69       |



### Decision Tree (TD-IDF)


As our basic model, we used the build up and analysis of our decision tree implementation to determine the approach to be followed by the subsequent models. 

We started with a decision tree classifier without weights and quickly understood the severity of the class imbalance of the data on our results. 

<img src="images/DT_unweighted.png" alt="unweighted classification report" width="700">

Hence, we moved to using weighted-f1 instead of macro-f1 and added confusion matrix and ROC-AUC charts for evaluation.

Our models are tuned using grid search and after tuning the weighted model, we can see that the decision tree accuracy and weighted f1 continues to drop. 

<img src="images/DT_perf.png" width="700">

However, for a model that hovers around 60% regardless, the performance of the model is rather poor. As expected, decision tree forms a good baseline to benchmark further models against.

For a deeper look into the code and more commentary on our decision tree build up, see: decisionTree.ipynb

### FFNN (TD-IDF)

Learning from decision tree, all our subsequent models immediately weighted and tuned with grid search. However, for deep learning models, it is now important for us address the issue of overfitting.

**FFNN (TD-IDF) Training and Validation Metrics**

<img src="images/FFNN_epoch.png" alt="unweighted classification report" width="700">

Given the loss profile over epochs for the model, we elect to stop training at epoch 6 to prevent overfitting. At epoch 6, we see weighted f1 and accuracy metrics have plateaued and while loss is low it is not at the level where it has overfitted which is evident at epoch 10 onwards. 

**FFNN (TD-IDF) Confusion Matrix**

<img src="images/FFNN_cm.png" alt="ffnn cm" width="700">

The performance of FFNN with a simple 3 layer model (chosen for comparability with LSTM who has 3 gates) with a batch size of 16 and hidden dimension of 128 performed a step up over decision trees as expected, almost touching 0.70 on both accuracy and weighted f1. Further, class seperability for FFNN was strong. From the confusion matrix, we observe that the diagonal performance is strong and where misclassifications occur, samples are flushed left, to bias label with the greatest samples, showing the lingering effects of imbalanced data. 

**FFNN (TD-IDF) ROC-AUC**

<img src="images/FFNN_rocauc.png" alt="ffnn rocauc" width="700">

ROC-AUC chart also indicates good class seperability with a floor of 0.79 and a ceiling of 0.91

### FFNN (GloVe)

Out of curiousity, we wondered how the vectorisation method ustilised would affect the performance of the FFNN model. We were curious to compare between the FFNN and LSTM architectures with the same embedding, even though for FFNN, we would be using one data point only per sample that is the average of all the embedding values for the article. While we do not expect FFNN with GloVe to perform well, we were curious about the effect size.

<img src="images/FFNN_G_epoch.png" width="700">

We can immediately observe the drop in performance with how the model is struggling to push past 0.5 in accuracy and weighted f1 values. The training loss is also starting to plataeu, indicating that further epochs would not help improve the performance of the ill designed model. 

<img src="images/FFNN_G_cm.png" width="700">

The confusion matrix shows that the model is as good as randomly guessing by predicting all over the spectrum for each actual label. 

<img src="images/FFNN_G_rocauc.png" width="700">

The ROC-AUC curve shows the same story with classifiers going closer to the diagonal line and even touching it. 

### LSTM (GloVe)

<img src="images/LSTM_epoch.png" width="700">

We chose GloVe over word2vec as the pretrained model has larger vocab more robust, more reasons why here, over us training our own word2vec embedding with our limited sample of n<5000 

We chose the 6B model with 100 dimensions as we are running our models on local and our compute power is limited. 

Applying GloVe to in a proper implementation to an LSTM model, there are immediate problems. First, even after fine tuning, the best model does not reach the baseline decision tree performance level of 0.60, instead it hovers around 0.50 even with extended training. 

In GridSearch, we configured max_epoch=20. However, the best params found stopped at epoch=13 due to early stop as weighted_f1 has not improved in the last 5 epoch (epoch 8-13). From the chart, we can see signs of overfitting for epoch 6. Validation loss is rapidly increasing while training loss is falling, and at the same time, weighted_f1 has plateaued while accuracy is increasing. This means that there is a trade off happening between accuracy with precision and recall as weighted_f1 is constant. 

At epoch 6, model has reached a good balance point. Training loss has decreased significantly from start value and validation metrics have not begun diverging significantly. Both accuracy and weighted f1 for training and validation remain reasonably aligned. Hence, we will stop training the model at epoch=6. 

LSTM Accuracy: 37.76%

LSTM Weighted F1-score: 0.4146

<img src="images/LSTM_cm.png" width="700">

Concerningly, the model shows some confusion between the extreme ends of the ideological positions (10 "right" instances misclassified as "left" and 46 left classified as right). "Right" predictions are also distributed across all classes, suggesting lower precision for this label. 

Final performance is worse than decision tree, which was contrary to expectations and disappointing.

**Commentary on LSTM vs FFNN**

The LSTM achieves 37.76% accuracy and a weighted F1-score of 0.4146, positioning its performance between the FFNN with TD-IDF (72.54% accuracy, 0.6109 weighted F1) and the FFNN with GloVe (32.27% accuracy, 0.3372 weighted F1). 

Comparing the implementations of FFNN and LSTM with the same embedding method of GloVe, as expected we can see that LSTM will outperform. FFNN took the average of embeddings while LSTM took in the data sequentially to learn temporal patterns in the data. 

Moving to BERT and Llama, we expect performance to increase for attention based models that account for semantic meaning and temporal patterns over purely sequence based LSTM. 

Comparing the better FFNN implementation with TD-IDF against LSTM with GloVe, unexpectedly, LSTM despite being a newer architecture, underperforms the classical method. 

Several reasons could have contributed to this. 

Firstly, we took a max sequence length of 128 because of memory issues (explained more in the next section). In this implementation, we inadvertantly tested our original hypothesis of if using only the first 100 words is enough to classify an article to the right label. This could potentially be unfair to compare with FFNN TD-IDF who analyses the entire article for vocabulary distributions. 

Secondly, we believe this is due to FFNN being able to generalize quickly on small data sets like we are using (<3000) while LSTM would need more data to learn the temporal patterns. 

**Commentary on LSTM Performance** 

Upon closer inspection, we notice the lack of performance is because LSTM is only 
taking the first 128 tokens (words in this case) for the article. 

While increasing the max sequence length might seem like an obvious solution, it presents significant computational challenges, even with our bidirectional architecture, both in compute (time) and space complexity scaling. It may also not be effective at all due to the inherent nature of the architecture that biases the memory cells to nearer words. 

**Computational complexity scaling**

Unlike Transformers which have O(n²) complexity with respect to sequence length due to their self-attention mechanism, LSTMs have a linear O(n) time complexity. However, this theoretical advantage becomes misleading when considering practical implementation constraints. 

For bidirectional LSTMs, the actual time complexity is O(2n), or simplified as O(n), but with a larger constant factor than unidirectional models
Memory requirements during training scale as O(n) as well, but the constant factors are substantial due to storing activations for backpropagation. 

**Space complexity considerations**

While Transformers have O(n²) space complexity for attention matrices, bidirectional LSTMs have O(n) space complexity but with significant constant factors:

Forward pass: O(n × d) where d is the hidden dimension size
Backward pass: An additional O(n × d) for gradient computation
Bidirectionality: Doubles both requirements to O(2n × d)

**Diminishing returns**

Khandelwal et al. (2018) in "Sharp Nearby, Fuzzy Far Away" (ACL) showed that LSTM language models primarily leverage information from the most recent 200 tokens, with diminishing contribution from tokens further back in the sequence with a sharp distinction between nearby context (most recent 50 tokens) and distant context. The model is sensitive to word order in the nearby context but largely ignores word order beyond 50 tokens, treating distant context more like a rough semantic or topical representation.

### BERT

For the BERT Model, we use the base BertModel from Hugging Face, and add an additional linear layer for classification. For tokenization, we use the supplied BertTokenizer from HuggingFace, which is based on WordPiece. BERT has achieved state of the art performance across a wide range of natural language understanding tasks, and has been shown to produce powerful contextualized embeddings of language in deeper layers of the model (Ethayarajh, 2019). There has also been studies showing that finetuning BERT can achieve or exceed performance of models published after it (Liu et al., 2019).

The goal of including BERT is to see how well the base model with some additional basic fine tuning using our dataset compared to the previous models we have tested. Note that our implementation differs from the cited papers, mainly because we wanted to see keep the model as close as we can to default to see how it compares to our previous models in terms of both performance and effort.

Note: We switch learning rate schedulers for transformers to a cosine decay schedule with a linear warmup. The curve of the schedule looks similar to this (though the numbers are not representative of what we used and are dependent on the early stopping mechanism, the dataset size, and other factors):

<img src="images/cosine_with_linear_warmup.png" width="700">

<img src="images/bert_perf.png" width="700">

Using the base dataset, we can see a divergence in loss, despite a decent f1 score on our validation set. Epoch 11 is where our validation weighted f1 score starts to dip and so we will use the model at that point for inference. However, the divergence in validation and training loss does suggest that there was some overfitting starting from epoch 4.

<img src="images/bert_cr.png" width="700">

BERT achieves 60.41% accuracy and a weighted F1-score of 0.6104, coming close to the of FFNN with TD-IDF (72.54% accuracy, 0.6109 weighted F1). It well outperformed LSTM, suggesting that BERT is able to contextualize text much better than a purely sequence based model like LSTM. Recall that FFNN with TD-IDF also strongly outperformed our base LSTM model - this does suggest that being able to contextualize is important in classification tasks.

Do note that the tokenizer for our implementation of BERT uses a length of 128, which could potentially be unfair to compare with FFNN TD-IDF who analyses the entire article for vocabulary distributions.


<img src="images/bert_cm.png" width="700">

Bert performs best on the "left" class, with 170/222 (76.6%) correct predictions, showing strong recognition of the dominant class.

Similar to LSTM, the model shows some confusion between distant ideological positions (18 "right" instances misclassified as "left"). "Right" predictions are distributed across all classes, suggesting lower precision for this label.

<img src="images/bert_roc.png" width="700">

ROC-AUC values for BERT are poor with a floor of 0.74 and a high of 0.87. Similar to the earlier commentary, it shows BERT performed about as good as FFNN TD-IDF and outperformed LSTM.

Overall, BERT performs as good as or better compared to our previous models. Most notably is that it performs close to FFNN TD-IDF, though FFNN TD-IDF still outperforms it. We do note that BERT takes significantly longer to train compared to previous models, and we did not perform any further fine tuning because of computational limits. Given unlimited resources, we would move deeper into testing different tokenizers and stretch the max_length variable for tokenized sequences further. Using Gridsearch for hyper parameter tuning was also not done in BERT's case, while it was done for our earlier models.

Overall, the results show that with little effort, BERT (and likely other models that can contextualize textual data). To further test our hypothesis, we will next move to an impolementation of the llama model, a newer - albeit more computationally more expensive - transformer model.

*
*Computational & Space complexity**

As mentioned, transformers have higher complexity when it comes to computation and space. However, the performance boost on a minimally tuned model does imply the trade off is worth it when it comes to implementation in BERTs case, since the benefits in performance should be offset by the relatively low training time when compared to newer more complex models such as Llama which we will see later.

Kawin Ethayarajh. 2019. How Contextual are Contextualized Word Representations? Comparing the Geometry of BERT, ELMo, and GPT-2 Embeddings. In Proceedings of the 2019 Conference on Empirical Methods in Natural Language Processing and the 9th International Joint Conference on Natural Language Processing (EMNLP-IJCNLP), pages 55–65, Hong Kong, China. Association for Computational Linguistics.

Liu, Y., Ott, M., Goyal, N., Du, J., Joshi, M., Chen, D., Levy, O., Lewis, M., Zettlemoyer, L. and Stoyanov, V., 2019. Roberta: A robustly optimized bert pretraining approach. arXiv preprint arXiv:1907.11692.

### Llama

Similar to BERT, we experiment the use of the Llama model on this classification task. Because of limited compute resources, we use TinlyLlama 1.1B in place of Llama, which is an optimized version of the Llama model. The TinyLLama is compact with only 1.1 billion parameters, which allows us more flexibility with implementation on our own laptops/desktops. Since Llama 2 and TinyLLama share the same tokenizer and architecture, we can also easily switch this out to using the Llama 2 model in this experiment.

For our implementation, we use the base TinyLlama-1.1B-Chat-v1 model, and add an extra classification layer for determining the labels to try to save on compute for an initial experiment on using the latest LLM models for classification.

Note: as mentioned above, the learning rate scheduler used for this is a cosine decay with a linear warmup.

<img src="images/llama_base_perf.png" width="700">

Training stopped at epoch 22, and notably much higher duration compared to previous models per epoch. While both train and validation loss continued to both trend downwards, the training stopped with the weighted f1 score stagnating. While performance on the training and validation sets does not look outstanding, it does suggest that more complex fine tuning might be beneficial to the model.

This however, will likely mean more time complexity and more compute needed.

Again, our parameters are very similar to our implementation of BERT, with the main exception of batch size, which we had to reduce due to compute reasons.

<img src="images/llama_base_cr.png" width="700">

In our findings, the model resulted in a relatively mediocre weighted f1 score at 0.5375 and decent accuracy at 51.72% on the test set, and thereafter performance started to drop. While this is better than LSTM but worst than BERT & FFNN TD-IDF, our validation loss was still trending downwards which does suggest we can fine tune the model further to achieve a higher performance. in our task.

<img src="images/llama_base_cm.png" width="700">

<img src="images/llama_base_roc.png" width="700">

The TinyLlama model shows very similar results in the confusion matrix as our previous models. The biggest difference was that previous models had issues with determining the minority label, while TinyLlama did not have as much of an issue.

The ROC curves also suggests that this model is able to generalise well, despite the class imbalance - which encouraged us to further optimize this model despite the higher computation cost.

Overall, TinyLlama showed some promising results in classification even with just a simple classification layer added to the base model. Given the availability of LoRA for fine tuning the model, this result prompted us to test the variation as well. 

**Compute and Space complexity**

The main limitation we face is the amount of compute needed for this model is significantly more than those we tested previously. This presents some challenges, one of which we mentioned was the reduction in our batch size in order for the training set to fit into memory. Another would be the amount of time needed to run this model, which is expected to increase when we augment our dataset with our scrapped data.

### Llama (LoRA)

Building on our previous TinyLlama findings we tried fine tuning with LoRA, which we expect to help our model converge faster and increase performance.

The reason we chose LoRA was its efficiency on compute and lower requirements compared to other fine tuning methods - studies found LoRA significantly improves model performance compared to fine-tuning only the classifier head, achieving state-of-the-art results on both simple and complex classification tasks while requiring fewer data samples. (Dausort et al. 2024)

<img src="images/llama_lora_perf.png" width="700">

The LoRA implementation found an optimal weighted f1 score at epoch 13 compared to epoch 22 in the base TinyLlama implementation. It was also able to achieve a higher performance.

We do see some signs of overfitting however, though the performance numbers for both accuracy and weighted f1 looks promising

Note: as mentioned above, the learning rate scheduler used for this is a cosine decay with a linear warmup..

<img src="images/llama_lora_cr.png" width="700">

The optimized TinyLlama model resulted in a weighted f1 score at 0.6369 and accuracy of 64.76%. This puts the model just behind of FFNN TD-IDF in our comparison.

<img src="images/llama_lora_cm.png" width="700">

<img src="images/llama_lora_roc.png" width="700">

Similarly, the LoRA tuned version showed better classification across classes as well in both the confusion matrix and ROC curves.

LoRA significatntly boosted the performance of the original TinyLlama model and had performance just shy of the FFNN TD-IDF implementation. The trade off however, is significantly more compute and time needed to train the model.

**Compute and Space complexity**

As per expectations, fine tuning with LoRA required longer duration and more memory than the base model. The performance did benefit however, which does somewhat justify the trade off in our opinion. Given the difference in time complexity and memory complexity between this model and BERT, however, it might be beneficial to use an optimized version of BERT as it may be able to perform as well with lower complexity.

Dausort, M., Godelaine, T., Zanella, M., Khoury, K.E., Salmon, I. and Macq, B., 2024. Exploring Foundation Models Fine-Tuning for Cytology Classification. arXiv preprint arXiv:2411.14975.

## Models - Run 2 (Augmented Dataset on the same parameters)

1. Decision Tree (TD-IDF) [combined_decisionTree.ipynb](combined_decisionTree.ipynb)
2. FFNN (TD-IDF) [combined_FFNN.ipynb](combined_FFNN.ipynb)
3. FFNN (GloVe) [combined_FFNN_GloVe.ipynb](combined_FFNN_GloVe.ipynb)
4. LSTM [combined_lstm.ipynb](combined_lstm.ipynb)
5. BERT [combined_bert.ipynb](combined_bert.ipynb)
6. Llama [combined_llama_base.ipynb](combined_llama_base.ipynb)
7. Llama LoRA [combined_llama_lora.ipynb](combined_llama_lora.ipynb)

| Model            | Accuracy | Weighted F1 | Accuracy (Aug) | Weighted F1 (Aug) |
|------------------|----------|-------------|----------------|-------------------|
| Decision Tree    | 59.27    | 60.29       | 56.25          | 57.08             |
| FFNN (TD-IDF)    | 69.79    | 68.85       | 66.60          | 65.20             |
| FFNN (GloVe)     | 32.27    | 33.72       | 33.59          | 36.11             |
| LSTM             | 37.76    | 41.46       | 66.51          | 67.08             |
| BERT             | 60.41    | 61.04       | 85.53          | 85.97             |
| TinyLlama        | 51.72    | 53.75       | 57.06          | 58.44             |
| TinyLlama (LoRA) | 64.76    | 63.69       | 85.41          | 85.14             |

<br>
<img src="images/draft.png" width="700">


### Overall Commentary 
Across the board we see a general improvement... (wait for ben)

### Decision Trees

**Decision Tree ROC-AUC**

Performance for key metrics of accuracy and weighted f1 for decision tree decreased. 

<img src="images/DT_rocauc.png" width="700">

**Decision Tree (Augmented) ROC-AUC**

<img src="images/DT_rocaucAug.png" width="700">

However, when looking into the ROC-AUC scores, we can see that there has been improved class seperability for the models. Hence although overall metrics dropped, the model is actually learning to generalise better with the additional data. 

### FFNN (TD-IDF) 

**FFNN (TD-IDF) Confusion Matrix**

<img src="images/FFNN_cm.png" width = "700">

**FFNN (TD-IDF) (Augmented) Confusion Matrix**

<img src="images/FFNN_cm_aug.png" width="700">

**FFNN (TD-IDF) ROC-AUC**

<img src="images/FFNN_rocauc.png" width="700">

**FFNN (TD-IDF) (Augmented) ROC-AUC**

<img src="images/FFNN_rocauc_aug.png" width="700">

Accuracy and weighted f1 also fell for FFNN (TD-IDF) though here we can see that the confusion matrix and ROC-AUC curves showed and overall degredation too. It is likely that the model now needs to be retuned (parameters are now sub optimal) as the old model did not have high generalisability to an expanded dataset. i.e. The old model was possibly overfitting to the now-expanded minority class samples. 

### FFNN (GloVe)

As expected, there was only a marginal improvement to the performance of FFNN with GloVe. This is as the model is fundamentally unsuited to using GloVe. Averaging out the word embeddings dilutes the important signals. It would require strong inherent differientials in the vocabulary used by each bias to predict well. From our EDA, we knew that is not that case. 

### LSTM (GloVe)

There is a massive improvement in the scores even with the modest addition to the dataset. This suggests that sequential information and word order are highly relevant features for this classification task. With the expanded dataset, LSTM could leverage the additional contextual information, while the FFNN did not capture these patterns from averaged embeddings.

Further, additional data is advantageous for the architecture of LSTMs as the LSTMs can utilise the memory and context cells to retain important information and forget irrelevant details, which is crucial for understanding text.

<img src="images/LSTM_epoch_aug.png" width="700">

With the same parameters as the earlier run, we can see that training can actually continue for longer with the augmented dataset as the model has not shown signs of strong overfitting while loss and performance metrics are all continuing to improve. 

**LSTM Confusion Matrix** 

<img src="images/LSTM_cm.png" width="700">

**LSTM Confusion Matrix (Augmented)** 

<img src="images/LSTM_cm_aug.png" width="700">

The addition of the new data helped the model to learn distinctions and the earlier issue of confusion between the extremes of the bias spectrum has been resovled. The diagonal performance is also strongest now though there remains some issues with flushing to the left (right predicted as left and left predicted as lean left)

**LSTM ROC-AUC**

<img src="images/LSTM_rocauc.png" width="700">

**LSTM ROC-AUC (Augmented)** 

<img src="images/LSTM_rocauc_aug.png" width="700">

ROC-AUC values for the LSTM rose from a floor of 0.63 to 0.87 and a ceiling of 0.72 to a ceiling of 0.89.  

The macro-average ROC-AUC also improved from 0.6768 to 0.8714 showing a much stronger learning of the class boundaries by the model. 

Even without retraining and retuning LSTM, we can see with the frozen parameters that class seperability has improved significantly. 

### BERT 

Similar to LSTM, BERT also experienced a significant boost to scores with the augmented dataset, suggesting that the base model can be further optimized as well.

**Base Dataset**

<img src="images/bert_perf.png" width="700">

**Augmented Dataset**

<img src="images/combined_bert_perf.png" width="700">

The model's training on augmented dataset shows less sign of overfitting compared to the base dataset. Performance was also significantly better.

**BERT Confusion Matrix (Base)** 

<img src="images/bert_cm.png" width="700">

**BERT Confusion Matrix (Augmented)** 

<img src="images/combined_bert_cm.png" width="700">

As with LSTM, the addition of the new data helped the model to learn distinctions and the earlier issue of confusion between the extremes of the bias spectrum has been resovled. The diagonal performance is also strongest now.

**BERT ROC-AUC (Base)**

<img src="images/bert_roc.png" width="700">

**BERT ROC-AUC (Augmented)** 

<img src="images/combined_bert_roc.png" width="700">

ROC-AUC values for the LSTM rose from a floor of 0.74 to 0.82 and a ceiling of 0.87 to a ceiling of 0.97.  

Without additional optimizations to our BERT implementation, we were still able to increase performance significantly by just augmenting our dataset.

### Llama

Surprisingly, the TinyLlama model didn't improve as significantly as we saw in BERT & LSTM. This might suggest that the LoRA parameter fine tuning is needed for Llama models to be used in classification tasks.

However, the results did show the model's ability to generalize better.

**Base Dataset**

<img src="images/llama_base_perf.png" width="700">

**Augmented Dataset**

<img src="images/combined_llama_base_perf.png" width="700">

The model's training on augmented dataset shows less sign of overfitting compared to the base dataset. Performance was also better, though not to the extent we see in LSTM and BERT. The continued drop in loss might suggest that it will be beneficial to extend the early stopping mechanism past 5 epochs, although we did not test this out in favor of moving on to the LoRA implementation, which we expect to yield better results.

**TinyLlama Confusion Matrix (Base)** 

<img src="images/llama_base_cm.png" width="700">

**TinyLlama Confusion Matrix (Augmented)** 

<img src="images/combined_llama_base_cm.png" width="700">

The model did well on both the base and augmented datasets on classifying each class, but had some problems across all the classes. While the run on the augmented dataset did better on some classes, it was a mixed result.

**TinyLlama ROC-AUC (Base)**

<img src="images/llama_base_roc.png" width="700">

**TinyLlama ROC-AUC (Augmented)** 

<img src="images/combined_llama_base_roc.png" width="700">

ROC-AUC values for the LSTM rose from a floor of 0.75 to 0.77 and a ceiling of 0.82 to a ceiling of 0.85.


### Llama

The LoRA tuning provided a performance boost with the augmented dataset more in line to what BERT and LSTM experienced. We do also note that the duration per epoch also increased significantly from 140s to 270s per epoch with a RTX 3060 GPU.

Despite the higher computational cost, the performance was close to BERT, which suggests that it may be more worthwhile to optimize a BERT model and fine tune it to the given task.

**Base Dataset**

<img src="images/llama_lora_perf.png" width="700">

**Augmented Dataset**

<img src="images/combined_llama_lora_perf.png" width="700">

The model's training on augmented dataset shows less sign of overfitting compared to the base dataset. Performance was also better, though not to the extent we see in LSTM and BERT. The continued drop in loss might suggest that it will be beneficial to extend the early stopping mechanism past 5 epochs, although we did not test this out in favor of moving on to the LoRA implementation, which we expect to yield better results. Also, the training phase went the full 30 epochs without triggering an early stop. This could mean that there is some chance we may be able to push for better performance with more epochs.

**TinyLlama Confusion Matrix (Base)** 

<img src="images/llama_lora_cm.png" width="700">

**TinyLlama Confusion Matrix (Augmented)** 

<img src="images/combined_llama_lora_cm.png" width="700">

The model did well on both the base and augmented datasets on classifying each class. The training on the augmented dataset allowed the model to generalize better, with the model correctly classifying more into the correct class during the augmented dataset run.

**TinyLlama ROC-AUC (Base)**

<img src="images/llama_lora_roc.png" width="700">

**TinyLlama ROC-AUC (Augmented)** 

<img src="images/combined_llama_lora_roc.png" width="700">

ROC-AUC values for the LSTM rose from a floor of 0.76 to 0.93 and a ceiling of 0.91 to a ceiling of 0.97.


## Limitations and Further Work

### Imbalanced Data 

Two routes were pursued to address the imbalanced data set issue. 

1. Scrapping additional data (as discussed above)
2. Generating synthetic data 

Regarding route two, we generated synthetic data by using chatgpt to experiment with on our LSTM model. There was a significant improvement in the model's performance, 

![syn perf](images/LSTM_perf_syn.png)

Accuracy rose from 0.6651 to 0.7368, while weighted F1 improved from 0.6708 to 0.7444 with the introduction of synthetic data. While these gains were not as pronounced as in the first augmentation phase, this aligns with our expectations—data augmentation often yields diminishing returns as the model saturates its capacity to benefit from additional training examples.

The improvements demonstrate that synthetic data helped the model generalize better, particularly on underrepresented classes. This suggests that even artificially generated samples can provide meaningful signals for learning decision boundaries, especially when real data is limited or imbalanced. Notably, the weighted F1 increase indicates better performance across class distributions, rather than improvements limited to majority classes.

Going forward, it might be more useful to focus on improving the quality of the synthetic data rather than only generating more of it so as to prevent overfitting ot synthetic patterns or noise. 

For example, using more controlled generation (e.g., class-conditional templates, prompt engineering, or filtering out low-quality outputs) could help.

However, we were unable to replicate the work with all our other data sets as running for the LSTM model alone took over 10 hours on local:

![synthetic data run took over 10 hours](images/SyntheticDataRun.png)

### Compute 

Surprisingly, we faced great challenges with compute even from the most basic models. Both group members have a Macbook Pro with an M1 chip, which we expected to be powerful enough to run the basic models quickly (under 5 mins). Instead our emphrical findings were just for training: 

| Model                            | Time |
|----------------------------------|------|
| Decision Tree                    | 18m  |
| FFNN (TD-IDF)                    | 26m  |
| FFNN (GloVe)                     | 28m  |
| LSTM                             | 47m  |
| BERT (one epoch only)            | 10m  |
| Llama (one epoch only)           | 20m  |
| Llama (LoRA) (one epoch only)    | 40m  |

We did have to resort running the transformer models on a borrow RTX 3060 to speed things up. While the GPU had 12GB of memory, it still ran into memory errors when running the Llama models. This resulted in the following times:

| Model                            | Time |
|----------------------------------|------|
| BERT                             | 19s  |
| Llama (one epoch only)           | 72s  |
| Llama (LoRA) (one epoch only)    | 140s |

### Overcoming Limitations 

We believe that our models performance will improve with more time. Both for more data to be scrapped and for the models to be run. We had minor successes running Llama on Google Colab before our compute ran out, and as proven earlier, the addition of the scrapped data to our base dataset improved performance for the newer models drastically (example LSTM  spiked from 37.76% to 66.51% for accuracy and 41.46 to 67.08 for weighted f1). 

# Conclusion

This project explored how the gamet of classic to modern machine learning techniques would perform in addressing the goal of classifying a news article into one of five bias labels. 

Our initial hypothesis was that we expected obvious improvements moving from an older architecture to the next. However, we were surprised to learn that classical methods can still hold their own in performance while also being faster and cheaper to train and run. 

On balance, for utilisation in production on a local device, we would suggest that a user adopts the FFNN with TD-IDF model as the pipeline is:

1. fast and lightweight
2. can take in the entire article over lstm/transformers who only take the first 128 tokens
3. performs best. 